# Notebook 01: Data Preprocessing and Augmentation

In this notebook, I will prepare the PlantDisease dataset for training. This involves defining transformations (augmentations) to help the model generalize better, and splitting the dataset into training and validation sets.

In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Setting up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Defining the path
data_dir = '../data/raw/PlantVillage'

Using device: cpu


### Step 1: Defining Transforms

I am applying random horizontal flips and rotations to the training set to prevent overfitting. The validation set will only be resized and normalized. I am using standard ImageNet normalization values because I will be fine-tuning a pre-trained ResNet model.

In [2]:
# Training transforms (includes augmentation)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transforms (no augmentation, just formatting)
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Loading the dataset twice to apply the different transforms
full_train_dataset = datasets.ImageFolder(data_dir, transform=train_transforms)
full_val_dataset = datasets.ImageFolder(data_dir, transform=val_transforms)

print(f"Total dataset size: {len(full_train_dataset)}")

Total dataset size: 41276


### Step 2: Creating the Train and Validation Splits

I am allocating 80% of the data for training the model and 20% for validating its accuracy. I will use `torch.utils.data.Subset` to create these splits based on randomized indices.

In [3]:
# Setting a seed for reproducibility
torch.manual_seed(42)

# Generating random indices
total_size = len(full_train_dataset)
indices = torch.randperm(total_size).tolist()

# Calculating the split size (80/20)
train_size = int(0.8 * total_size)

# Creating the final training and validation subsets
train_data = Subset(full_train_dataset, indices[:train_size])
val_data = Subset(full_val_dataset, indices[train_size:])

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")

# Creating DataLoaders to feed the data in batches during training
batch_size = 32

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

print("DataLoaders created successfully.")

Training set size: 33020
Validation set size: 8256
DataLoaders created successfully.
